<a href="https://colab.research.google.com/github/KinzaAsif2456/discoverey/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Supervised Opportunity Scoring & Learning-to-Rank:
We frame the task as a direct learning-to-rank / scoring task where the model attempts to estimate a continuous decay risk score
 for each published page. Pages are ordered descending by predicted opportunity score to maximize the recovery of lost organic search traffic per editorial hour spent.

In [11]:
import os, sys, subprocess


IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [12]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.head)


<bound method NDFrame.head of                  content_id          client_id  search_volume  competition  \
0      content_304f48230142  client_f369cb89fc           10.0         0.67   
1      content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2      content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3      content_331d6c4de07b  client_19581e27de           10.0         0.00   
4      content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   
...                     ...                ...            ...          ...   
29995  content_c322796023c8  client_e29c9c180c           10.0         0.05   
29996  content_526572edb3fa  client_7f2253d7e2            0.0         0.00   
29997  content_38112bdd0c6e  client_349c41201b           10.0         1.00   
29998  content_ab26273a7e7a  client_19581e27de           10.0         0.00   
29999  content_887020f20b5e  client_6208ef0f77            0.0         0.00   

      competition_level   cpc    

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Proxy: There's no ground truth for "should this page be refreshed" — that would need an actual experiment (refresh it, see if traffic recovers). So is_declining_label is a proxy: a page whose search demand measurably dropped in the most recent month is a reasonable stand-in for "worth an editor's attention," even though decline and refresh-worthiness aren't strictly the same thing.

In [13]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(df["is_declining_label"].value_counts())
print(f"{df['is_declining_label'].mean()*100:.1f}% of pages are labeled declining")

is_declining_label
1    16262
0    13738
Name: count, dtype: int64
54.2% of pages are labeled declining


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I'm using Precision@50 because the two error types aren't symmetric for this decision. A false positive near the top of the queue wastes real editor hours this week. A false negative just means that page waits — it's still visible next week's re-scoring. Precision@50 penalizes exactly the error that's expensive (bad picks at the top), not the one that's recoverable.

In [14]:
naive_rank = df.sort_values("search_volume", ascending=False).head(50)
precision_at_50_naive = naive_rank["is_declining_label"].mean()
print(f"Precision@50 sorting by search_volume alone: {precision_at_50_naive:.2f}")
print(f"Overall decline rate (what a random top-50 would get): {df['is_declining_label'].mean():.2f}")

Precision@50 sorting by search_volume alone: 0.42
Overall decline rate (what a random top-50 would get): 0.54


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [15]:
cols = ["content_id", "client_id", "content_type", "days_since_last_update",
        "impressions_90d", "trend_pct", "trend_direction", "is_declining_label"]
df[cols].head(10)

,content_id,client_id,content_type,days_since_last_update,impressions_90d,trend_pct,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,20,3803,-41.4,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,25,15320,-57.7,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,20,12581,-60.9,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,22,11751,-13.8,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,14,19140,-34.7,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,20,3970,-38.9,down,1
6,content_9a34b442b552,client_8722616204,keyword article,20,20,-92.3,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,22,1724,0.6,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,20,32574,-58.8,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,104,1240,-29.2,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A staleness rule ("flag anything not updated in 180+ days") barely overlaps with what's actually declining: only 17 pages are both stale and still highly visible, while 13,152 pages (43.8%) are declining despite still getting real search demand. One threshold on one column misses almost the whole problem.

In [16]:
rule_flag = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
print("Pages flagged by staleness rule:", rule_flag.sum())
print("Of those, share actually declining:", df.loc[rule_flag, "is_declining_label"].mean())
print("Overall decline rate:", df["is_declining_label"].mean())

Pages flagged by staleness rule: 17
Of those, share actually declining: 0.9411764705882353
Overall decline rate: 0.5420666666666667


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.